# Preprocessing data in MNE-Python

```
Modified by:
Maximilien Chaumon

From authors:
Marijn van Vliet, Britta Westner, Alexandre Gramfort, Denis A. Engemann
```

## Setup

We start out with loading the packages we need. These include `matplotlib` for plotting, `os` for path management, `numpy` for numerical computations, and of course `mne`.
We also use matplotlib "magic" to ask for figures to be plotted inline. 

In [ ]:
# This makes figures appear inside the notebook instead of in a new window.
%matplotlib inline

# These are the Python packages we will use in this notebook.
import os # for handling file paths
import numpy as np  # for numerical computations
import matplotlib.pyplot as plt  # for making custom figures
import mne  # for MEG and EEG data analysis

Let's double check your MNE-Python version. This should give back `1.10.1` or higher.

In [ ]:
mne.__version__

We set the log-level of MNE-Python to 'warning' so the output is less verbose:

In [ ]:
mne.set_log_level('warning')

## Set the path to the data

You should have downloaded the `ds000117-pruned` folder. We have to let Python know, where to find this folder on your disk. You will have to adjust the path below to reflect your computer and path structure!
You can print the whole path and check if the directory is correct.

In [ ]:
# Change the following path to where the folder ds000117 is on your disk.
data_path = "../ds000117_pruned"  # `./` means the folder of this notebook

# Change the following path to where you unzipped the extra data (`extra_meg_data-v2.zip`) on your disk.
extra_path = "../extra_data_mne"  # `./` means the folder of this notebook

# Based on the `data_path` you specified above, this is where the raw MEG+EEG data should be.
raw_fname = f"{data_path}/derivatives/meg_derivatives/sub-01/ses-meg/meg/sub-01_ses-meg_task-facerecognition_run-01_proc-sss_meg.fif"

# issue error if raw_fname does not exist say it's ok otherwise
if not os.path.exists(raw_fname):
    raise FileNotFoundError(f"Could not find the raw data file at {raw_fname}. Please check the path and try again.")
else:
    print("Raw data file found. Proceeding with loading the data.")

## Access and read the raw data

In [ ]:
raw = mne.io.read_raw(raw_fname, preload=False)
print(raw)

In [ ]:
raw

## Explore the data file


Now let's look at the measurement info. It can give details about:

   - sampling rate
   - filtering parameters
   - available channel types
   - bad channels
   - etc.

In [ ]:
print(raw.info)

<div class="alert alert-success">
    <b>Exercise</b>:
     <ul>
    <li>How many channels do you have for each type of sensors?</li>
    <li>What is the sampling frequency?</li>
    <li>Have the data been filtered?</li>
    <li>What is the frequency of the line noise?</li>
    <li>Is there any bad channel?</li>
    </ul>
</div>

In [ ]:
raw.ch_names[:10]  # this prints the first ten channels

In [ ]:
raw.plot_sensors(ch_type="grad");

Now let's look at the EEG channels.

In this data, the position of EEG sensors was digitized with a 3D pen. In this case, in order to get the best looking figure, we will first fit a sphere to the digitized points.

In [ ]:
radius, center, _ = mne.bem.fit_sphere_to_headshape(raw.info, dig_kinds="eeg")
sphere = tuple(center) + (radius,)
raw.plot_sensors(ch_type="eeg", sphere=sphere);

## Setting channel types and re-referencing

Some channels are wrongly defined as EEG in the file. 
Two of these are EOG (EEG061 and EEG062) and EEG063 is actually an ECG channel. EEG064 was recording but not connected to anything, so we'll make it `"misc"` instead. 
We will now set the channel types for those wrongly classified channels. This will be useful for automatic artifact rejection.

In [ ]:
raw.set_channel_types({"EEG061": "eog",  # actually EOG not EEG
                       "EEG062": "eog",  # actually EOG not EEG
                       "EEG063": "ecg",  # actually ECG not EEG
                       "EEG064": "misc"})  # EEG064 free-floating electrode

# we also rename the EOG and ECG channels:
raw.rename_channels({"EEG061": "EOG061",
                     "EEG062": "EOG062",
                     "EEG063": "ECG063"})

In [ ]:
raw.plot_sensors(kind='topomap', ch_type='eeg', sphere=sphere);

After we have fixed the channels, we can now compute an average reference for the EEG. 

In [ ]:
raw.info

In [ ]:
# For setting the reference, we have to load the data into memory:
raw.load_data()

# before re-referencing, data is referenced to the nose (not shown).
raw.copy().pick_types(eeg=True).plot()

# now let's re-reference
raw.set_eeg_reference(ref_channels=['EEG001'])
raw.copy().pick_types(eeg=True).plot()

raw.set_eeg_reference(ref_channels='average')
raw.copy().pick_types(eeg=True).plot()

## Accessing the data

To access the data just use the `[]` syntax as to access any element of a list, dict etc. Note that `raw[]` returns two things: the data and the times array.

In [ ]:
start, stop = 0, 1000
data, times = raw.copy().pick_types(meg='mag')[:, start:stop]  # fetch all channels and the first 10 time points
print(data.shape)
print(times.shape)

In [ ]:
plt.plot(times, data.T)
plt.xlabel("Time (s)")
plt.ylabel("Amplitude")

## Resampling the data

We will now change the sampling frequency of the data to speed up the computations.

In [ ]:
raw.load_data()  # load data into memory
raw.resample(300)

And let's remove unecessary channels - some empty stimulus channels, misc. channels, and HPI channels.

In [ ]:
to_drop = ["STI201", "STI301", "MISC201", "MISC202", "MISC203",
           "MISC204", "MISC205", "MISC206", "MISC301", "MISC302",
           "MISC303", "MISC304", "MISC305", "MISC306", "CHPI001",
           "CHPI002", "CHPI003", "CHPI004", "CHPI005", "CHPI006",
           "CHPI007", "CHPI008", "CHPI009"]
           
raw.drop_channels(to_drop)

## Filtering the data and plotting raw data

To see what effect filtering has for our data, let's quickly look at our data first! For full functionality, we ask matplotlib to show the plot in a separate window.

In [ ]:
# Tell MNE-Python to use the fast QT data browser (should be the default)
%matplotlib qt
mne.viz.set_browser_backend("qt")
raw.plot()

### Inspecting the spectral contents of the data

We are now going to plot the power spectrum. The PSD plot is interactive: select frequency bands to see its distribution scross sensors.

In [ ]:
%matplotlib inline
psd = raw.compute_psd()
psd.plot(sphere=sphere);

We want to filter the data between 0 and 40 Hz using a linear-phase finite-impulse response (FIR) filter.

In [ ]:
raw_filtered = raw.copy().filter(0, 40)
psd_filtered = raw_filtered.compute_psd()
psd_filtered.plot(sphere=sphere);

Now that we filtered our data, let's look at it again. Can you spot the difference?

In [ ]:
raw.plot()
raw_filtered.plot()

## Look at the event structure of the data

The data has different events, which mark which stimulus was shown to the participants. The event/trigger structure is as follows:
- 5, 6, 7: famous faces
- 13, 14, 15: unfamiliar faces
- 17, 18, 19: scrambled faces

We first look at which events are there:

In [ ]:
events = mne.find_events(raw, stim_channel='STI101', verbose=True)

We look at the first 10 events.
Three columns are:
- event onset sample
- event duration
- event value (code)

In [ ]:
print(events[:10])

Let's visualize the paradigm:

In [ ]:
events = events[events[:, 2] < 20]  # take only events with code less than 20
%matplotlib inline

mne.viz.plot_events(events, raw.info["sfreq"]);

For event trigger and conditions we use a Python dictionary with keys that contain "/" for grouping sub-conditions

In [ ]:
event_id = {
    "face/famous/first": 5,
    "face/famous/immediate": 6,
    "face/famous/long": 7,
    "face/unfamiliar/first": 13,
    "face/unfamiliar/immediate": 14,
    "face/unfamiliar/long": 15,
    "scrambled/first": 17,
    "scrambled/immediate": 18,
    "scrambled/long": 19,
}

In [ ]:
mne.viz.plot_events(events, sfreq=raw.info["sfreq"], event_id=event_id);

We can now re-visit our raw data plot:

In [ ]:
%matplotlib qt

raw.plot(event_id=event_id, events=events);

## Epoch data and artifact rejection

Define epochs parameters:

In [ ]:
tmin = -0.5  # start of each epoch (500ms before the trigger)
tmax = 2.0  # end of each epoch (2000ms after the trigger)

Define the baseline period:

In [ ]:
baseline = (-0.2, 0)  # means from 200ms before to stim onset (t = 0)

We also pick channels now - MEG, EEG and EOG channels

In [ ]:
picks = mne.pick_types(raw.info, meg=True, eeg=True, eog=True,
                       stim=False, exclude='bads')

The easiest (and maybe also most dangerous?) way to clean your data is to define peak-to-peak (amplitude range) rejection parameters for gradiometers, magnetometers and EOG.

<div class="alert alert-info">
    <b>REMARK</b>:
     <ul>
    <li>The <a href="https://autoreject.github.io/">autoreject project</a> aims to solve this problem of reject parameter setting. See the <a href="https://www.sciencedirect.com/science/article/pii/S1053811917305013">paper</a> for more info.</li>
    </ul>
</div>

In [ ]:
reject = dict(grad=4000e-13, mag=4e-12, eog=150e-6)  # this can be highly data dependent

Now we can put all of this together and create epochs:

In [ ]:
epochs = mne.Epochs(raw, events, event_id, tmin, tmax, proj=True,
                    picks=picks, baseline=baseline,
                    reject=reject)

epochs.load_data()  # load data in memory

Accounting for projector delay

During stimulus presentation, there was a delay of 34.5 ms between the event marker and the projector actually showing the image.
We need to correct the events accordingly.

In [ ]:
epochs.shift_time(0.034) # shift epochs by 34ms to compensate for trigger delay

In [ ]:
print(epochs)  # let's look at some details about the epochs object

In [ ]:
epochs.plot(events=events, event_id=event_id);

Let's explicitly drop the epochs we identified as _bad_ through the thresholds we identified above:

In [ ]:
epochs_nobad = epochs.copy()
epochs_nobad.drop_bad()  # remove bad epochs based on reject
print(epochs_nobad)  # let's look at some details about the epochs object

## A closer look at artifact rejection


See how epochs were dropped

In [ ]:
%matplotlib inline

epochs_nobad.plot_drop_log();

We just **loose half of our epochs due to EOG artifacts**!

We can probably do better. Let's use independant component analysis (ICA) to remove components related to EOG and other offenders, ie., ECG.

Here is the workflow, we'll first detect EOG artifacts and visualize their impact. Then we'll use ICA to mitigate these artifacts.

In [ ]:
raw_highpass = raw.copy().filter(1, None)

In [ ]:
# There is a function to create EOG epochs:
eog_epochs = mne.preprocessing.create_eog_epochs(raw_highpass)
eog_epochs.average().plot_joint(topomap_args=dict(sphere=sphere));

Let's see where those EOG segments show up in our raw data:

In [ ]:
%matplotlib qt
raw.plot(events=eog_epochs.events);

In [ ]:
event_id

In [ ]:
# stack events and eog_events
all_events = np.vstack((events, eog_epochs.events))
# call eog_events "blink" in the event_id dictionary
event_id["blink"] = 998  # we can use any code that is not already
event_colors = {"blink": "r",
                "face/famous/first": "g",
                "face/famous/immediate": "g",
                "face/famous/long": "g",
                "face/unfamiliar/first": "b",
                "face/unfamiliar/immediate": "b",
                "face/unfamiliar/long": "b",
                "scrambled/first": "m",
                "scrambled/immediate": "m",
                "scrambled/long": "m"}
event_id
raw.plot(event_id=event_id, events=all_events, duration=60., event_color=event_colors);

Let's perform ICA on the raw data. It needs data that is well-centered on 0, so we high-pass first.
To speed things up, only fit enough components to explain 95% of the data.

In [ ]:
ica = mne.preprocessing.ICA(0.95).fit(raw_highpass)
ica.save(f"{extra_path}/sub-01-ica.fif", overwrite=True)

%matplotlib inline
ica.plot_components();

In [ ]:
bads_eog, eog_scores = ica.find_bads_eog(raw_highpass)
ica.plot_scores(eog_scores, bads_eog);

In [ ]:
ica.plot_properties(eog_epochs, bads_eog);

Now the important question is how many components one should keep? Tip: some of them don't look like clear artifact patterns. 

The good news is that we don't need to decide __*right*__ now - as you could see the projectors are stored with the data, but inactive at the moment.

BUT: let's repeat this procedure for the ECG, i.e. heart beat artifacts

In [ ]:
# same business, same issue for ECG
ecg_epochs = mne.preprocessing.create_ecg_epochs(raw_highpass)
ecg_epochs.average().plot_joint(topomap_args=dict(sphere=sphere));

In [ ]:
bads_ecg, ecg_scores = ica.find_bads_ecg(raw_highpass)
ica.plot_scores(ecg_scores, bads_ecg);
ica.plot_properties(ecg_epochs, bads_ecg);

We can see that we also face contamination from the cardiac signal... we'll project that out as well.

## Apply ICA and visualize effect

Let's mark the detected EOG and ECG components for removal and apply the ICA to the epoched data.

In [ ]:
ica.exclude = bads_eog + bads_ecg
epochs_clean = ica.apply(epochs.copy())

Let's look at one frontal MEG channel before and after applying the ICA:

In [ ]:
epochs.plot_image(picks='MEG0142', sigma=1.);
epochs_nobad.plot_image(picks='MEG0142', sigma=1.);
epochs_clean.plot_image(picks='MEG0142', sigma=1.);

In [ ]:
eog_epochs.plot_image(picks='MEG0142', sigma=1.);
ica.apply(eog_epochs.copy()).plot_image(picks='MEG0142', sigma=1.);


In [ ]:

ecg_epochs.plot_image(picks='MEG0142', sigma=1.);
ica.apply(ecg_epochs.copy()).plot_image(picks='MEG0142', sigma=1.);



In [ ]:
# from scipy.signal import hilbert, butter, sosfilt

# def create_sort_function(sort_by='phase', timepoint=0.5):
#     """
#     Create a sorting function for epochs.
    
#     Parameters
#     ----------
#     sort_by : str
#         'phase' to sort by phase at timepoint, 'amplitude' to sort by amplitude
#     timepoint : float
#         Time point (in seconds) at which to sort
#     """
#     def sort_epochs(times, data):
#         # Find the time point closest to timepoint
#         idx = np.argmin(np.abs(times - timepoint))
        
#         if sort_by == 'phase':
#             # Bandpass filter around 10Hz
#             sfreq = 1 / (times[1] - times[0])  # sampling frequency
#             sos = butter(4, [4, 6], btype='band', fs=sfreq, output='sos')
#             data_filtered = sosfilt(sos, data, axis=-1)
            
#             # Get analytic signal to extract phase
#             analytic = hilbert(data_filtered, axis=-1)
#             sort_values = np.angle(analytic[:, idx])
#             indices = np.argsort(sort_values)
#             # new figure to show the sorted epochs
#             plt.figure()
#             plt.imshow(data[indices,:], aspect='auto', origin='lower', extent=[times[0], times[-1], 0, data.shape[0]], cmap='RdBu_r')
#             plt.colorbar()
#             plt.xlabel('Time (s)')
#             plt.ylabel('Epoch')
#             # new figure to show the sorted epochs
#             plt.figure()
#             plt.imshow(data_filtered[indices,:], aspect='auto', origin='lower', extent=[times[0], times[-1], 0, data_filtered.shape[0]], cmap='RdBu_r')
#             plt.colorbar()
#             plt.xlabel('Time (s)')
#             plt.ylabel('Epoch')
#         elif sort_by == 'amplitude':
#             # Sort by amplitude at timepoint
#             sort_values = data[:, idx]
#             indices = np.argsort(sort_values)
#             # new figure to show the sorted epochs
#             plt.figure()
#             plt.imshow(data[indices,:], aspect='auto', origin='lower', extent=[times[0], times[-1], 0, data.shape[0]], cmap='RdBu_r')
#             plt.colorbar()
#             plt.xlabel('Time (s)')
#             plt.ylabel('Epoch')
#         else:
#             raise ValueError("sort_by must be 'phase' or 'amplitude'")
        
#         return indices
    
#     return sort_epochs

# Use 'phase' or 'amplitude' here
# sort_func = create_sort_function(sort_by='amplitude', timepoint=1.5)
# epochs_clean.plot_image(picks='MEG0123', sigma=1., order=sort_func);
# epochs_clean.plot_image(picks='MEG0123', sigma=1.);

#### Some thoughts on artifact rejection

We now tackled the artifacts in this data set by computing SSP projections. There are many other ways to do artifact rejection:

- mark artifacts by hand (visual inspection)
- use thresholds (which failed on this dataset!)
- use ICA
- use an automated pipeline, e.g. the <a href="https://autoreject.github.io/">autoreject project</a>
- ...

The best recommendation is: get to know your (raw) data!

## Save Epochs

The standard way is to save the epochs as a `.fif` file together with all the header data. Epochs are saved with the suffix `-epo.fif`.

In [ ]:
epochs_fname = raw_fname.replace('_meg.fif', '-epo.fif')  # create the file name
epochs_fname

In [ ]:
epochs_clean.save(epochs_fname, overwrite=True) 

## Bonus: Visualizing epochs data

See [this page](https://mne.tools/stable/auto_tutorials/epochs/20_visualize_epochs.html) for options on how to visualize epochs.

In [ ]:
# We have already looked at the epochs in a stacked plot:
epochs_clean.plot_image(picks='EEG065', sigma=1.);

We can also look at the epochs in a data browser window:

In [ ]:
%matplotlib qt
epochs_clean.plot();